In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [2]:
file = np.load("../../data/T1492_x1151_y1_z127_c2.npz")
data = file['timeseries']

In [3]:
class VelocityDataset(Dataset):
    def __init__(self, data):
        # data: numpy array of shape (t, x, z, v)
        self.data = torch.tensor(data, dtype=torch.float32).permute(0, 3, 1, 2)
        # Now shape: (t, v=2, x=1151, z=127)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
dataset = VelocityDataset(data)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [5]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.ConvTranspose2d(32, 2, kernel_size=3, stride=2, padding=1, output_padding=0)
        )

    def forward(self,x):
        latent=self.encoder(x)
        #print(latent.shape)
        recon=self.decoder(latent)
        return recon

In [6]:
model = ConvAutoencoder().to('cuda')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 20

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0
    
    for batch in loader:
        batch = batch.to('cuda')
        
        optimizer.zero_grad()
        
        recon_batch = model(batch)
        #print(recon_batch.shape)
        #print(batch.shape)
        loss = criterion(recon_batch,batch)   # reconstruction loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.size(0)

    print(f"Epoch {epoch+1}/{n_epochs} - Loss: {total_loss / len(loader.dataset):.6f}")

Epoch 1/20 - Loss: 0.145726
Epoch 2/20 - Loss: 0.006575
Epoch 3/20 - Loss: 0.004480
Epoch 4/20 - Loss: 0.003711
Epoch 5/20 - Loss: 0.003133
Epoch 6/20 - Loss: 0.002761
Epoch 7/20 - Loss: 0.002522
Epoch 8/20 - Loss: 0.002396
Epoch 9/20 - Loss: 0.002187
Epoch 10/20 - Loss: 0.002079
Epoch 11/20 - Loss: 0.001915
Epoch 12/20 - Loss: 0.001978
Epoch 13/20 - Loss: 0.001739
Epoch 14/20 - Loss: 0.001788
Epoch 15/20 - Loss: 0.001606
Epoch 16/20 - Loss: 0.001587
Epoch 17/20 - Loss: 0.001491
Epoch 18/20 - Loss: 0.001519
Epoch 19/20 - Loss: 0.001542
Epoch 20/20 - Loss: 0.001367


In [7]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

vx_all = data[..., 0]
vz_all = data[..., 1]

vx_min, vx_max = np.min(vx_all), np.max(vx_all)
vz_min, vz_max = np.min(vz_all), np.max(vz_all)

norm_x = Normalize(vmin=vx_min, vmax=vx_max)
norm_z = Normalize(vmin=vz_min, vmax=vz_max)

def plot_vx_vz(vx, vz):

    plt.figure(figsize=(10,8))

    # --- vx ---
    plt.subplot(2,1,1)
    plt.imshow(vx.T, origin='lower', aspect='auto', cmap='viridis', norm=norm_x)
    plt.colorbar(label='vx')
    plt.title(f"vx")
    plt.xlabel("x-coordinate")
    plt.ylabel("z-coordinate")

    # --- vz ---
    plt.subplot(2,1,2)
    plt.imshow(vz.T, origin='lower', aspect='auto', cmap='plasma', norm=norm_z)
    plt.colorbar(label='vz')
    plt.title(f"vz")
    plt.xlabel("x-coordinate")
    plt.ylabel("z-coordinate")

    plt.tight_layout()
    plt.show()


In [11]:
model.eval()
mse_list = []

with torch.no_grad():
    for x_true in dataset:
        #vx_true = x_true[0]
        #vz_true = x_true[1]
        #plot_vx_vz(vx_true, vz_true)
        x_true = x_true.unsqueeze(0).to('cuda')
        x_pred = model(x_true).squeeze(0).cpu()
        #vx_pred = x_pred[0]
        #vz_pred = x_pred[1]
        #plot_vx_vz(vx_pred, vz_pred)
        #vx_diff = (vx_true - vx_pred)**2
        #vz_diff = (vz_true - vz_pred)**2
        mse_list.append(np.mean((x_true.to('cpu').numpy() - x_pred.numpy())**2))
        #plot_vx_vz(vx_diff, vz_diff)
        #print(np.sum(vz_diff.numpy()))
        #print(np.sum(vz_diff.numpy()))
        #print()
        ##

In [13]:
mse_list = np.array(mse_list)

In [15]:
np.mean(mse_list)

np.float32(0.0011444758)